# Chapitre 9 · L'attention (exercice)

Notebook du chapitre 9 de *Construire un LLM de zéro*. Le cœur conceptuel du
livre : la self-attention dérivée pas à pas, le scaled dot-product attention
en NumPy puis en PyTorch, le facteur d'échelle, le masque causal, et la classe
`SelfAttention` complète.

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
exécutable de bout en bout. Lis, exécute, modifie pour voir. Ensuite, la
section **Exercices** : c'est LÀ que tu écris TON attention, trou par trou,
validée par des `assert`. **Le pacte IA débranchée s'applique à ces exercices
en entier** : ferme l'onglet de ton assistant IA, littéralement. Aucune IA
n'écrit ce code à ta place ; elle relit, vérifie et débloque une fois les
asserts passés, elle n'écrit pas.

Tout tourne sur CPU, hors ligne, en quelques secondes. Aucun GPU nécessaire.

## Setup

NumPy pour dérouler les calculs à la main, PyTorch pour la version finale, matplotlib pour regarder les matrices d'attention.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

print("NumPy", np.__version__, "| PyTorch", torch.__version__)

## 1. Un modèle qui n'écoute pas encore

Le modèle du chapitre 8 lit une fenêtre figée et mélange les embeddings avec des
doses fixées par position : il ne peut pas décider, en lisant la phrase, où regarder.
Ce chapitre construit le mécanisme qui manque : chaque mot choisit lui-même quels
autres mots écouter. C'est l'attention, la pièce centrale de tous les LLMs modernes.

## 2. L'attention, dérivée à la main

La phrase fil rouge du chapitre : *« Awa pose la mangue sur la table parce qu'elle est mûre. »*
On en garde cinq mots et on leur donne des embeddings jouets de dimension 4, fabriqués à la main :
« elle » et « mangue » pointent dans des directions proches.

In [ ]:
mots = ["la", "mangue", "est", "mûre", "elle"]
E = np.array([
    [0.1, 0.1, 0.0, 0.0],   # la
    [1.0, 0.2, 0.1, 1.0],   # mangue
    [0.0, 0.1, 0.1, 0.0],   # est
    [0.5, 0.1, 0.2, 0.6],   # mûre
    [0.8, 0.1, 0.0, 0.9],   # elle
], dtype=np.float32)
print("E :", E.shape, "(un embedding de dimension 4 par mot)")

### 2.1 Première idée : moyenner le contexte

Le nouveau vecteur de « elle » est la moyenne uniforme des embeddings. Mieux que
rien, mais « mangue » y pèse exactement autant que « la » : l'information est diluée.

In [ ]:
# Première idée : le nouveau vecteur de « elle » = moyenne UNIFORME des embeddings.
moyenne = E.mean(axis=0)
print("Moyenne naïve :", np.round(moyenne, 3))
print("Chaque mot pèse 1/5 = 0.2 : « mangue » autant que « la ». L'information est diluée.")

### 2.2 Deuxième idée : doser selon la pertinence

Le produit scalaire du chapitre 2 mesure la pertinence entre deux embeddings ;
le softmax du chapitre 5 transforme les scores en doses positives qui somment à 1.

In [ ]:
# Deuxième idée : doser selon la pertinence. Produit scalaire, puis softmax.
scores_elle = E @ E[4]                       # similarité entre « elle » et chaque mot
poids = np.exp(scores_elle - scores_elle.max())
poids = poids / poids.sum()
for mot, p in zip(mots, poids):
    print(f"  {mot:8s} alpha = {p:.3f}")
print()
print("Nouveau vecteur de « elle » :", np.round(poids @ E, 3))
print("« mangue » domine le mélange : l'attention vient de naître.")

### 2.3 Troisième idée : séparer les rôles

Query (ce que je cherche), key (comment je suis indexé), value (ce que je transmets) :
trois projections linéaires apprises du même embedding. Leur code arrive avec la
classe `SelfAttention` de la section 5 ; d'abord, la formule.

## 3. La formule : le scaled dot-product attention

### 3.1 Une ligne, quatre étapes

$\mathrm{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \mathrm{softmax}(\mathbf{Q}\mathbf{K}^{\top} / \sqrt{d_k})\, \mathbf{V}$.
Vérifions le mini-exemple déroulé à la main dans le livre : 2 tokens (« elle », « mangue »),
$d_k = d_v = 2$, avec des Q, K, V **distincts** (Q ≠ K : ce que « elle » cherche n'est pas ce
par quoi elle se laisse trouver). On part des embeddings X = identité, puis on projette.

In [ ]:
# Le mini-exemple de la section 3.1 : les mêmes nombres que le calcul à la main.
X_mini = np.array([[1., 0.], [0., 1.]])          # ligne 0 = elle, ligne 1 = mangue
W_Q = np.array([[0., 1.], [1., 0.]])             # elle cherche ce que mangue offre
W_K = np.array([[1., 0.], [0., 1.]])             # key = identite
W_V = np.array([[10., 0.], [0., 20.]])           # values contrastees

Q_mini = X_mini @ W_Q                            # [[0, 1], [1, 0]]
K_mini = X_mini @ W_K                            # [[1, 0], [0, 1]]
V_mini = X_mini @ W_V                            # [[10, 0], [0, 20]]
assert not np.allclose(Q_mini, K_mini), "Q et K doivent differer : c'est la separation des roles."

scores_mini = Q_mini @ K_mini.T / np.sqrt(2)     # etapes 1 + 2
poids_mini = np.exp(scores_mini - scores_mini.max(axis=1, keepdims=True))
poids_mini = poids_mini / poids_mini.sum(axis=1, keepdims=True)   # etape 3
output_mini = poids_mini @ V_mini                # etape 4

print("Scores mis a l'echelle :\n", np.round(scores_mini, 3))
print("Poids d'attention (ligne 0 'elle' : [0.33, 0.67]) :\n", np.round(poids_mini, 2))
print("Output (ligne 0 'elle' : [3.30, 13.40], teintee de mangue) :\n", np.round(output_mini, 2))

### 3.2 Le facteur d'échelle : la preuve empirique

Le chapitre l'affirme (section 3.2) : sans la division par $\sqrt{d_k}$, les scores se dispersent
avec la dimension et le softmax sature. Mesurons-le avec l'**entropie** des poids : haute quand
l'attention est répartie, proche de zéro quand elle se fige sur un seul token.

In [ ]:
def entropie_moyenne(weights, eps=1e-10):
    """Entropie de Shannon moyenne des lignes de la matrice de poids."""
    return -(weights * torch.log(weights + eps)).sum(dim=-1).mean().item()

dims = [8, 16, 32, 64, 128, 256]
ent_sans, ent_avec = [], []
for d in dims:
    xd = torch.randn(5, 10, d)
    s = torch.matmul(xd, xd.transpose(-2, -1))
    ent_sans.append(entropie_moyenne(torch.softmax(s, dim=-1)))
    ent_avec.append(entropie_moyenne(torch.softmax(s / np.sqrt(d), dim=-1)))

plt.figure(figsize=(8, 4))
plt.plot(dims, ent_sans, "o-", label="sans / √d_k")
plt.plot(dims, ent_avec, "s-", label="avec / √d_k")
plt.xlabel("d_k")
plt.ylabel("entropie moyenne des poids")
plt.title("Sans scaling, l'attention se fige quand la dimension grandit")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Sans scaling : l'entropie s'effondre (softmax piqué). Avec : elle reste stable.")

### Cas qui échoue : softmax saturé quand on oublie le `/ √d_k`

À `d_k = 512`, sans la division, un poids frôle 1 et la « pente » locale de la softmax
(`a * (1 - a)`) tombe à zéro : plus de gradient utile, le modèle n'apprendrait plus.

In [ ]:
# Cas qui échoue PUIS correction. Gardé sous try/except : le notebook ne plante jamais.
try:
    torch.manual_seed(0)
    d_grand, n = 512, 8
    Qg, Kg = torch.randn(1, n, d_grand), torch.randn(1, n, d_grand)

    def sensibilite(w):
        # pente locale de la softmax : a*(1-a) ; si un poids tend vers 1, elle tend vers 0
        return (w * (1 - w)).sum(dim=-1).mean().item()

    s_sans = torch.matmul(Qg, Kg.transpose(-2, -1))            # PAS de / sqrt(d_k)
    w_sans = torch.softmax(s_sans, dim=-1)
    s_avec = s_sans / np.sqrt(d_grand)
    w_avec = torch.softmax(s_avec, dim=-1)

    print(f"d_k = {d_grand}")
    print(f"SANS scaling : poids max = {w_sans.max():.4f} | sensibilité = {sensibilite(w_sans):.6f}  (saturé, gradient mort)")
    print(f"AVEC scaling : poids max = {w_avec.max():.4f} | sensibilité = {sensibilite(w_avec):.6f}  (le gradient circule)")
except Exception as e:
    print(f"Démo interrompue : {type(e).__name__}: {e}")

### 3.3 En NumPy, pas à pas

On reprend les données jouets du chapitre : 4 tokens, `d_k = 4`. K est l'identité (chaque key
isole un trait) et Q est fabriquée pour que le token 0 « matche » fortement la key 0.
On veut voir l'attention choisir le bon livre de la bibliothèque.

In [ ]:
# 4 tokens, d_k = 4 : valeurs choisies à la main pour l'intuition
Q = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0.5, 0.5, 0, 0], [0, 0, 0.5, 0.5]], dtype=np.float32)
K = np.eye(4, dtype=np.float32)                      # chaque key isole un trait
V = np.array([[10, 0, 0, 0], [0, 20, 0, 0], [0, 0, 30, 0], [0, 0, 0, 40]], dtype=np.float32)
d_k = Q.shape[-1]
print("Q :", Q.shape, "| K :", K.shape, "| V :", V.shape, "| d_k =", d_k)

Les quatre étapes, exactement le code de la section 3.3 du chapitre : scores mis à
l'échelle, stabilité numérique, softmax ligne par ligne, moyenne pondérée des values.

In [ ]:
d_k = Q.shape[-1]
scores = Q @ K.T / np.sqrt(d_k)                      # Étapes 1 + 2 : scores mis à l'échelle
print("scores mis à l'échelle (√d_k = 2, la case (0,0) vaut 0.5) :\n", scores)

scores -= scores.max(axis=1, keepdims=True)          # stabilité numérique (voir 3.4)
weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)  # Étape 3 : softmax
output = weights @ V                                  # Étape 4 : moyenne pondérée

print("weights (chaque ligne somme à 1) :\n", np.round(weights, 3))
print("output = weights @ V :\n", np.round(output, 2))
print("Ligne 0 de weights :", np.round(weights[0], 3), ": le poids le plus fort tombe sur la key 0.")

### 3.4 La stabilité numérique du softmax

`exp(1000)` déborde en flottant et le softmax naïf part en `NaN`. La parade, le
**log-sum-exp trick** : soustraire le maximum de chaque ligne avant l'exponentielle.
Mathématiquement identique, numériquement indestructible.

In [ ]:
# Cas qui échoue PUIS correction : le mauvais résultat est calculé, jamais d'exception.
scores_geants = np.array([[1000.0, 999.0]])

with np.errstate(over="ignore", invalid="ignore"):   # on met les warnings en sourdine
    exp_naif = np.exp(scores_geants)                 # exp(1000) -> inf : overflow
    poids_naif = exp_naif / exp_naif.sum(axis=1, keepdims=True)
print("Softmax naïf   :", poids_naif, " <- inf / inf = NaN, l'entraînement mourrait ici")

decale = scores_geants - scores_geants.max(axis=1, keepdims=True)   # le plus grand devient 0
poids_stable = np.exp(decale) / np.exp(decale).sum(axis=1, keepdims=True)
print("Softmax stable :", np.round(poids_stable, 3), " <- [0.73, 0.27], le calcul à la main du chapitre")

## 4. Le masque causal

Notre LLM prédit le token suivant : le token en position $t$ ne doit pas voir les positions
futures, sinon il apprendrait à copier. Le masque est triangulaire inférieur :
1 = autorisé, 0 = interdit. Regarde le triangle : le présent regarde le passé, jamais l'avenir.

In [ ]:
masque6 = np.tril(np.ones((6, 6)))
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(masque6, cmap="Greens", vmin=-0.3, vmax=1.3)
for i in range(6):
    for j in range(6):
        ax.text(j, i, int(masque6[i, j]), ha="center", va="center")
ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xlabel("Key (position écoutée)")
ax.set_ylabel("Query (position qui écoute)")
ax.set_title("Masque causal : le présent regarde le passé, jamais l'avenir")
plt.tight_layout()
plt.show()

L'implémentation de la section 4.3 du chapitre, en PyTorch : $-\infty$ sur le futur,
**avant** le softmax. Après l'exponentielle, ces positions pèsent exactement zéro.

In [ ]:
torch.manual_seed(0)
scores = torch.randn(4, 4)                               # des scores jouets, seq_len = 4

seq_len = scores.size(-1)
mask = torch.tril(torch.ones(seq_len, seq_len))          # tril = triangulaire inférieure
scores = scores.masked_fill(mask == 0, float('-inf'))    # -inf sur le futur
weights = torch.softmax(scores, dim=-1)                  # exp(-inf) = 0 -> poids nul

print("Scores masqués :\n", scores)
print("Poids après softmax :\n", np.round(weights.numpy(), 3))
print("Sommes par ligne (toujours 1) :", weights.sum(dim=-1))

La même mécanique en NumPy, emballée en une fonction complète : les quatre étapes,
plus le masque optionnel appliqué sur les **scores** avec `np.where(mask == 0, -np.inf, scores)`.

In [ ]:
def masque_causal_lecon(n):
    """Masque causal (n, n) : 1 sur le triangle inférieur (passé + présent), 0 sur le futur."""
    return np.tril(np.ones((n, n), dtype=np.float32))

def attention_lecon(Q, K, V, mask=None):
    """Scaled dot-product attention complète, en NumPy. Retourne (output, weights)."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)                       # étapes 1 + 2
    if mask is not None:
        scores = np.where(mask == 0, -np.inf, scores)     # masque AVANT le softmax
    scores = scores - scores.max(axis=1, keepdims=True)   # log-sum-exp
    exp_scores = np.exp(scores)
    weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)   # étape 3
    output = weights @ V                                  # étape 4
    return output, weights

out_c, w_c = attention_lecon(Q, K, V, mask=masque_causal_lecon(4))
print("Poids causaux (triangle supérieur nul, lignes qui somment à 1) :\n", np.round(w_c, 3))
print("Output causal, ligne 0 = V[0] : le token 0 ne voit que lui-même.\n", np.round(out_c, 2))

### Cas qui échoue : masque appliqué *après* le softmax

Le piège du chapitre (section 4.3) : `softmax(scores) * mask` retire de la masse de probabilité
sans renormaliser. Les lignes ne somment plus à 1, la moyenne pondérée est biaisée.

In [ ]:
# Cas qui échoue PUIS correction. Gardé sous try/except : le notebook ne plante jamais.
try:
    torch.manual_seed(1)
    n = 4
    scores_t = torch.randn(n, n)
    causal = torch.tril(torch.ones(n, n))

    w_avant = torch.softmax(scores_t.masked_fill(causal == 0, float("-inf")), dim=-1)  # BON
    w_apres = torch.softmax(scores_t, dim=-1) * causal                                 # MAUVAIS

    print("Somme des poids par ligne (doit valoir 1.0) :")
    print("ligne | masque AVANT (correct) | masque APRES (faux)")
    for i in range(n):
        print(f"  {i}   |        {w_avant[i].sum():.4f}        |      {w_apres[i].sum():.4f}")
    print()
    print("Masque APRES : sommes différentes de 1, la moyenne pondérée des V perd de la masse.")
    print("Correction : toujours masked_fill(..., -inf) AVANT la softmax, qui renormalise pour nous.")
except Exception as e:
    print(f"Démo interrompue : {type(e).__name__}: {e}")

## 5. La cible : l'attention PyTorch complète

Le code exact de la section 5 du chapitre : la même logique en gérant en plus le **batch**,
Q, K, V en `(B, n, d_k)`. `transpose(-2, -1)` n'échange que les deux dernières dimensions,
la dimension de batch traverse tout sans qu'on s'en occupe. C'est cette fonction et la
classe qui suit que tu réécriras de tes mains dans les exercices, IA débranchée.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)   # (B, n, n)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))      # masque AVANT softmax
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

torch.manual_seed(0)
Qb, Kb, Vb = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
out, w = scaled_dot_product_attention(Qb, Kb, Vb)
print("output :", tuple(out.shape), "| weights :", tuple(w.shape))
ref = F.scaled_dot_product_attention(Qb, Kb, Vb)
print("Colle à la référence de PyTorch :", torch.allclose(out, ref, atol=1e-5))

### La preuve que tout est différentiable

La promesse de la section 3.5 du chapitre. On compare les gradients de `.backward()`
(ceux que ton moteur du chapitre 4 calculerait) à des **différences finies** : on pousse
chaque entrée de $\pm\varepsilon$ et on mesure de combien la sortie bouge. Si les deux
coïncident, la rétropropagation traverse l'attention sans accroc.

In [ ]:
def perte_attention(x):
    out, _ = scaled_dot_product_attention(x, x, x)   # self-attention : Q = K = V = x
    return out.sum()

x = torch.randn(1, 3, 4, dtype=torch.float64, requires_grad=True)
perte_attention(x).backward()
grad_backward = x.grad.clone()

eps = 1e-6
grad_num = torch.zeros_like(x)
with torch.no_grad():
    flat, num_flat = x.view(-1), grad_num.view(-1)
    for i in range(flat.numel()):
        old = flat[i].item()
        flat[i] = old + eps
        f_plus = perte_attention(x).item()
        flat[i] = old - eps
        f_moins = perte_attention(x).item()
        flat[i] = old
        num_flat[i] = (f_plus - f_moins) / (2 * eps)

ecart = (grad_backward - grad_num).abs().max().item()
print(f"Écart max backward vs différences finies : {ecart:.2e} (sous 1e-4 : les gradients collent)")
print("L'attention est différentiable de bout en bout : la rétropropagation la traverse.")

### La classe `SelfAttention`

Il ne reste qu'à emballer la fonction avec les trois projections apprises de la section 2.3 :
un même `x`, trois lectures. C'est le module qu'on branchera dans le Transformer au chapitre 10.

In [ ]:
class SelfAttention(nn.Module):
    """Self-attention : Q, K, V sont trois projections linéaires apprises du même x."""

    def __init__(self, d_model):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        Q = self.W_Q(x)                     # tout vient du même x : « self »-attention
        K = self.W_K(x)
        V = self.W_V(x)
        return scaled_dot_product_attention(Q, K, V, mask)

torch.manual_seed(1)
attn = SelfAttention(d_model=16)
x = torch.randn(2, 5, 16)
out, w = attn(x, mask=torch.tril(torch.ones(5, 5)))
print("output :", tuple(out.shape), "| weights :", tuple(w.shape))
out.sum().backward()
print("Gradient sur W_Q :", attn.W_Q.weight.grad is not None,
      ": la descente de gradient peut apprendre les trois projections.")

## 6. Self-attention, cross-attention

Même formule partout ; ce qui change, c'est d'où viennent Q, K et V. Self-attention :
Q, K, V de la même séquence (notre LLM, GPT). Cross-attention : Q d'une séquence,
K et V d'une autre (la traduction de Bahdanau). Pas de code nouveau ici : la classe
de la section 5 couvre notre cas, celui qu'il faut maîtriser à fond.

## 7. Lire une matrice d'attention

La matrice des poids se lit comme une heatmap : chaque ligne est une distribution de probabilité.
Ici on fabrique des Q et K avec un signal diagonal fort : chaque token se regarde surtout lui-même.

In [ ]:
rng = np.random.default_rng(0)
n_viz, d_viz = 6, 8
Q_viz = rng.standard_normal((n_viz, d_viz)).astype(np.float32)
K_viz = rng.standard_normal((n_viz, d_viz)).astype(np.float32)
for i in range(n_viz):
    Q_viz[i, i] += 2.0      # signal fort sur la dimension i : motif diagonal
    K_viz[i, i] += 2.0

S_viz = Q_viz @ K_viz.T / np.sqrt(d_viz)
W_viz = np.exp(S_viz - S_viz.max(axis=1, keepdims=True))
W_viz = W_viz / W_viz.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(W_viz, cmap="YlOrRd")
for i in range(n_viz):
    for j in range(n_viz):
        ax.text(j, i, f"{W_viz[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(n_viz), [f"K{j}" for j in range(n_viz)])
ax.set_yticks(range(n_viz), [f"Q{i}" for i in range(n_viz)])
ax.set_xlabel("Key (token écouté)")
ax.set_ylabel("Query (token qui écoute)")
ax.set_title("Qui regarde qui : chaque ligne est une distribution")
fig.colorbar(im, label="Poids d'attention")
plt.tight_layout()
plt.show()

## Exercices

On y est : le passage IA débranchée du livre (l'autre était le moteur d'autograd du
chapitre 4). Sept exercices qui suivent l'ordre de construction du chapitre : les
scores, le softmax stable, la moyenne pondérée, le masque causal, l'attention complète
en NumPy, puis la version PyTorch et la classe `SelfAttention`. Les niveaux ● à ●●●
indiquent l'effort attendu. Chaque cellule marquée `# TODO(toi)` contient un ou
plusieurs trous ; complète, puis exécute la cellule de validation (`assert`) qui
suit : si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » s'applique ici en entier** : ferme l'onglet de ton
assistant, littéralement. Aucune IA n'écrit ces fonctions à ta place. Bloqué ?
Relis la section correspondante du chapitre, elle contient tout ; l'idéal est même
d'écrire chaque fonction sans recopier la leçon. Les réponses sont dans le notebook
solution, mais chaque ligne recopiée sans l'avoir cherchée est une ligne que tu ne
posséderas pas.

### Exercice 1 · Les scores mis à l'échelle — niveau ●

Étapes 1 et 2 de la formule : le produit scalaire de toutes les paires query/key, divisé par
$\sqrt{d_k}$. Rappel du réflexe `shape` : `(n, d_k) @ (d_k, n) = (n, n)`.

In [ ]:
# Les données jouets de la leçon : 4 tokens, d_k = 4.
Q = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0.5, 0.5, 0, 0], [0, 0, 0.5, 0.5]], dtype=np.float32)
K = np.eye(4, dtype=np.float32)
V = np.array([[10, 0, 0, 0], [0, 20, 0, 0], [0, 0, 30, 0], [0, 0, 0, 40]], dtype=np.float32)

def scores_attention(Q, K):
    """Étapes 1 + 2 : scores de toutes les paires, mis à l'échelle.

    Q : (n, d_k), K : (n, d_k)  ->  scores : (n, n)
    L'élément (i, j) se lit : « à quel point le token i s'intéresse au token j ».
    """
    d_k = Q.shape[-1]
    # TODO(toi) : calcule Q @ K.T puis divise par np.sqrt(d_k)
    scores = ...
    return scores

In [ ]:
S = scores_attention(Q, K)
S_attendu = np.array([[0.500000, 0.000000, 0.000000, 0.000000],
              [0.000000, 0.500000, 0.000000, 0.000000],
              [0.250000, 0.250000, 0.000000, 0.000000],
              [0.000000, 0.000000, 0.250000, 0.250000]], dtype=np.float32)
assert S.shape == (4, 4), f"shape {S.shape} au lieu de (4, 4) : as-tu bien transposé K ?"
assert np.allclose(S, S_attendu, atol=1e-4), "valeurs inattendues : as-tu divisé par np.sqrt(d_k) et pas d_k ?"
print("Exercice 1 validé : les scores sont corrects.")
print(S)

### Exercice 2 · Le softmax stable, ligne par ligne — niveau ●●

Étape 3 : chaque ligne de scores devient une distribution de probabilité. Et souviens-toi de la
section 3.4 du chapitre : on soustrait le maximum de chaque ligne avant l'exponentielle
(le log-sum-exp trick), sinon `exp` déborde sur de grands scores.

In [ ]:
def softmax_stable(scores):
    """Étape 3 : softmax ligne par ligne, numériquement stable.

    scores : (n, n)  ->  weights : (n, n), chaque ligne positive et de somme 1.
    """
    # TODO(toi), en trois temps :
    #   1. retranche à chaque ligne son maximum        (.max(axis=1, keepdims=True))
    #   2. exponentie                                  (np.exp)
    #   3. divise chaque ligne par sa somme            (.sum(axis=1, keepdims=True))
    weights = ...
    return weights

In [ ]:
W = softmax_stable(S)
W_attendu = np.array([[0.354661, 0.215113, 0.215113, 0.215113],
              [0.215113, 0.354661, 0.215113, 0.215113],
              [0.281088, 0.281088, 0.218912, 0.218912],
              [0.218912, 0.218912, 0.281088, 0.281088]], dtype=np.float32)
assert np.allclose(W.sum(axis=1), 1.0, atol=1e-5), "chaque ligne doit sommer à 1"
assert np.allclose(W, W_attendu, atol=1e-4), "valeurs inattendues : softmax appliqué ligne par ligne (axis=1) ?"
assert W[0].argmax() == 0, "le token 0 devrait regarder surtout la key 0 (son meilleur match)"
grands = softmax_stable(np.array([[1000.0, 1001.0, 999.0]]))
assert np.isfinite(grands).all(), "overflow : as-tu soustrait le max de chaque ligne avant np.exp ?"
print("Exercice 2 validé : softmax correct et stable, même sur des scores énormes.")
print(np.round(W, 3))

### Exercice 3 · La moyenne pondérée des values — niveau ●

Étape 4 : chaque token compose son nouveau vecteur en mélangeant les values de tous les tokens,
chacune pondérée par l'attention qu'il lui porte.

In [ ]:
def melange_values(weights, V):
    """Étape 4 : output = weights @ V.

    weights : (n, n), V : (n, d_v)  ->  output : (n, d_v)
    """
    # TODO(toi) : une seule ligne
    output = ...
    return output

In [ ]:
O = melange_values(W, V)
O_attendu = np.array([[3.5466, 4.3023, 6.4534, 8.6045],
              [2.1511, 7.0932, 6.4534, 8.6045],
              [2.8109, 5.6218, 6.5674, 8.7565],
              [2.1891, 4.3782, 8.4326, 11.2435]], dtype=np.float32)
assert O.shape == (4, 4), f"shape {O.shape} au lieu de (4, 4)"
assert np.allclose(O, O_attendu, atol=1e-2), "valeurs inattendues : c'est weights @ V, dans cet ordre"
print("Exercice 3 validé : la moyenne pondérée des values est correcte.")
print(np.round(O, 2))

### Exercice 4 · Fabriquer le masque causal — niveau ●

`np.tril` (« triangular lower ») fabrique la matrice triangulaire inférieure.

In [ ]:
def masque_causal(n):
    """Masque causal (n, n) : 1 sur le triangle inférieur (passé + présent), 0 sur le futur."""
    # TODO(toi) : une seule ligne, avec np.tril et np.ones
    masque = ...
    return masque

In [ ]:
M = masque_causal(4)
assert M.shape == (4, 4), f"shape {M.shape} au lieu de (4, 4)"
assert np.allclose(M, np.tril(np.ones((4, 4)))), "le triangle inférieur (diagonale comprise) doit valoir 1"
assert M[0, 1] == 0 and M[3, 0] == 1, "1 = autorisé (passé), 0 = interdit (futur)"
print("Exercice 4 validé : le masque causal est correct.")
print(M.astype(int))

### Exercice 5 · Tout assembler : l'attention complète en NumPy — niveau ●●

Les quatre étapes plus le masque optionnel. Règle d'or du chapitre (section 4.3) : le masque
s'applique sur les **scores**, avec $-\infty$, **avant** le softmax. `np.where(mask == 0, -np.inf, scores)`
fait l'affaire : après l'exponentielle, ces positions pèsent exactement zéro.

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention complète, en NumPy.

    Retourne (output, weights) : output (n, d_v), weights (n, n).
    """
    # TODO(toi), avec tes fonctions précédentes :
    #   1. scores = scores_attention(Q, K)
    #   2. si mask n'est pas None : mets -np.inf sur les positions où mask == 0 (AVANT le softmax)
    #   3. weights = softmax_stable(scores)
    #   4. output = melange_values(weights, V)
    ...
    return output, weights

In [ ]:
out, w = attention(Q, K, V)
assert np.allclose(out, O_attendu, atol=1e-2), "sans masque, on doit retrouver l'output de l'exercice 3"

out_c, w_c = attention(Q, K, V, mask=masque_causal(4))
OC_attendu = np.array([[10.0000, 0.0000, 0.0000, 0.0000],
              [3.7754, 12.4492, 0.0000, 0.0000],
              [3.5987, 7.1973, 8.4080, 0.0000],
              [2.1891, 4.3782, 8.4326, 11.2435]], dtype=np.float32)
assert np.allclose(w_c.sum(axis=1), 1.0, atol=1e-5), "chaque ligne doit encore sommer à 1"
assert np.allclose(w_c[np.triu_indices(4, k=1)], 0.0, atol=1e-6), "le futur doit peser exactement zéro"
assert np.allclose(out_c[0], V[0], atol=1e-4), "le token 0 ne voit que lui-même : son output = V[0]"
assert np.allclose(out_c, OC_attendu, atol=1e-2), "valeurs inattendues avec le masque causal"
print("Exercice 5 validé : attention complète, causale, et correctement renormalisée.")
print(np.round(w_c, 3))

### Exercice 6 · La fonction `scaled_dot_product_attention` — niveau ●●

La même logique qu'en NumPy, en gérant en plus le **batch** : Q, K, V sont en `(B, n, d_k)`.
`transpose(-2, -1)` n'échange que les deux dernières dimensions, la dimension de batch
traverse tout sans qu'on s'en occupe. C'est le code exact de la section 5 du chapitre ;
la validation compare ta version à la référence de PyTorch.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention en PyTorch, avec batch et masque optionnel.

    Q, K, V : (B, n, d_k)  ->  output (B, n, d_v), weights (B, n, n)
    """
    d_k = Q.shape[-1]
    # TODO(toi), les quatre étapes :
    #   1. scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    #   2. si mask n'est pas None : scores.masked_fill(mask == 0, float('-inf'))
    #   3. weights = torch.softmax(scores, dim=-1)
    #   4. output = torch.matmul(weights, V)
    ...
    return output, weights

In [ ]:
torch.manual_seed(0)
Qb, Kb, Vb = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)

out, w = scaled_dot_product_attention(Qb, Kb, Vb)
assert out.shape == (2, 5, 8) and w.shape == (2, 5, 5), f"shapes : {out.shape}, {w.shape}"
ref = F.scaled_dot_product_attention(Qb, Kb, Vb)
assert torch.allclose(out, ref, atol=1e-5), "sans masque, ton résultat doit coller à la référence de PyTorch"

masque = torch.tril(torch.ones(5, 5))
out_c, w_c = scaled_dot_product_attention(Qb, Kb, Vb, mask=masque)
ref_c = F.scaled_dot_product_attention(Qb, Kb, Vb, is_causal=True)
assert torch.allclose(out_c, ref_c, atol=1e-5), "avec masque causal, idem"
assert torch.allclose(w_c.sum(dim=-1), torch.ones(2, 5), atol=1e-5)
print("Exercice 6 validé : ton attention PyTorch colle à F.scaled_dot_product_attention.")

### Exercice 7 · La classe `SelfAttention` — niveau ●●●

Il ne reste qu'à emballer ta fonction avec les trois projections apprises de la section 2.3 :
un même `x`, trois lectures. C'est le module qu'on branchera dans le Transformer au chapitre 10.

In [ ]:
class SelfAttention(nn.Module):
    """Self-attention : Q, K, V sont trois projections linéaires apprises du même x."""

    def __init__(self, d_model):
        super().__init__()
        # TODO(toi) : trois projections nn.Linear(d_model, d_model, bias=False)
        # nommées self.W_Q, self.W_K, self.W_V
        ...

    def forward(self, x, mask=None):
        # TODO(toi) : projette x en Q, K, V puis renvoie scaled_dot_product_attention(Q, K, V, mask)
        ...

In [ ]:
torch.manual_seed(1)
attn = SelfAttention(d_model=16)
x = torch.randn(2, 5, 16)
masque = torch.tril(torch.ones(5, 5))

out, w = attn(x, mask=masque)
assert out.shape == (2, 5, 16) and w.shape == (2, 5, 5), f"shapes : {out.shape}, {w.shape}"
assert torch.allclose(w.sum(dim=-1), torch.ones(2, 5), atol=1e-5), "chaque ligne doit sommer à 1"
assert torch.allclose(w[:, 0, 1:], torch.zeros(2, 4), atol=1e-6), "causale : le token 0 ne voit que lui-même"

out.sum().backward()
assert attn.W_Q.weight.grad is not None and torch.isfinite(attn.W_Q.weight.grad).all(), \
    "les gradients doivent circuler jusqu'aux projections"
print("Exercice 7 validé : ta SelfAttention est complète, causale et entraînable.")
print("Pacte tenu. Tu viens d'écrire le cœur des LLMs.")

## Où en est notre LLM ?

Sept validations vertes : pacte tenu. Tu viens d'écrire, de tes mains :

- les **scores** query/key mis à l'échelle par $\sqrt{d_k}$ ;
- le **softmax stable** (log-sum-exp) ;
- la **moyenne pondérée** des values ;
- le **masque causal**, appliqué avant le softmax ;
- la fonction **`scaled_dot_product_attention`** et la classe **`SelfAttention`**,
  complètes, différentiables, prêtes pour un vrai modèle.

Une matmul, un softmax, une matmul : c'est littéralement ce qui propulse les LLMs.

Au **chapitre 10**, on assemble le Transformer autour de cette pièce : multi-head attention,
encodage positionnel, connexions résiduelles, normalisation, FFN, et on entraîne le tout.